# Climate incentives and biodiversity complements

At 5% biodiversity headroom, the minimum number of national biodiversity complements is 20 with one shared climate coefficient. Allowing seven distinct climate coefficients attains the national minimum of four. This notebook reproduces the original policy comparisons, the exact shared-price frontier and the economic mechanism behind the binding common-price restriction.

Run the cells in order. Quick calculations use released numerical certificates. Full coefficient reproduction reconstructs the source database and evaluates all 13 alternatives in each of 87,644 modeled spatial units across 23 EU countries. The final cell downloads one compact archive of numerical outputs and all five publication figures.
The Belgian economic safeguard sensitivity is shown separately, with unresolved bounds retained. Set `RUN_FULL_SOURCE_REPLAY = True` in the source calculation cell to reproduce the entire coefficient-level analysis.


In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, shutil, subprocess, sys, tempfile, time, zipfile

REPOSITORY_URL = "https://github.com/calinadriancomes/climate-biodiversity-complements.git"
REPOSITORY_REF = "main"
LOCAL_REPOSITORY = None  # Optional path to an extracted repository.
REPOSITORY_ZIP = None    # Optional path to a complete repository ZIP.
in_colab = "google.colab" in sys.modules

root = Path(LOCAL_REPOSITORY).expanduser().resolve() if LOCAL_REPOSITORY else next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "code/reproduce.py").is_file() and (p / "inputs/SOURCE.json").is_file()), None)

if root is None:
    checkout = Path(tempfile.mkdtemp(prefix="climate_biodiversity_"))
    if REPOSITORY_ZIP is None and in_colab:
        candidate = checkout / "climate-biodiversity-complements"
        try:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
                            REPOSITORY_URL, str(candidate)], check=True,
                           capture_output=True, text=True, timeout=180)
            root = candidate
        except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
            from google.colab import files
            print("Repository retrieval was unavailable. Upload its complete ZIP file.")
            uploaded = files.upload()
            archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
            if len(archives) != 1:
                raise ValueError("Upload one complete repository ZIP.")
            REPOSITORY_ZIP = archives[0]
    if root is None and REPOSITORY_ZIP is not None:
        with zipfile.ZipFile(REPOSITORY_ZIP) as archive:
            for member in archive.infolist():
                target = (checkout / member.filename).resolve()
                if not target.is_relative_to(checkout.resolve()):
                    raise ValueError("The ZIP contains a path outside its extraction directory.")
            archive.extractall(checkout)
        candidates = [p.parent.parent for p in checkout.glob("**/code/reproduce.py")
                      if (p.parent.parent / "inputs/SOURCE.json").is_file()]
        if len(candidates) != 1:
            raise ValueError("The ZIP must contain exactly one complete repository.")
        root = candidates[0]
    if root is None:
        raise FileNotFoundError("Open the notebook from the extracted repository, or set LOCAL_REPOSITORY or REPOSITORY_ZIP.")

if not (root / "code/verify_matrices.py").is_file():
    raise FileNotFoundError("The selected repository is incomplete.")
output = root / "outputs"
output.mkdir(parents=True, exist_ok=True)
print("Repository:", root.name)


In [ ]:
from importlib.metadata import PackageNotFoundError, version
required = dict(line.strip().split("==") for line in (root / "requirements.txt").read_text().splitlines()
                if line.strip() and not line.lstrip().startswith("#"))
missing = []
for name, expected in required.items():
    try:
        if version(name) != expected:
            missing.append(name)
    except PackageNotFoundError:
        missing.append(name)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(root / "requirements.txt")], check=True)
for name, expected in required.items():
    if version(name) != expected:
        raise RuntimeError(f"Required dependency: {name} {expected}")
# Separate processes use the installed dependency versions even if a notebook
# service imported another numerical-library version before setup.
def run(script, *arguments):
    subprocess.run([sys.executable, "-B", str(root / script), *map(str, arguments)],
                   check=True, cwd=root)
print("Specified numerical dependencies available.")


## Quick mathematical checks

The input identities fix the numerical technology and its scientific certificates. Small finite examples separately test complete response sets, persistent ties, isolated breakpoint failures, action-independent payment recentering and the participation property of nonnegative payments. These examples are mathematical checks, not additional European observations.


In [ ]:
started = time.perf_counter()
run("code/check.py", "--output", output / "checks")
print(f"Quick checks: {time.perf_counter() - started:.1f} seconds")


## National accounts and the original policy constructions

Recalculate the tables from country-level numerical records. Monetary sums use 2015 US dollars per year. The climate index contains a stock term and a conventional 20-year methane component. National biodiversity scores are not added across countries.

The two original complete constructions attain different ecological outcomes. Their transfers therefore do not identify the fiscal cost of harmonization. The subsequent source calculation independently verifies the underlying assignments and their outcomes.


In [ ]:
run("code/consolidate.py", "--output", output)
run("code/tables.py", "--data", output / "data", "--output", output / "tables")
with (output / "data/policies.csv").open(newline="") as stream:
    national_records = list(csv.DictReader(stream))
for policy in ["National climate coefficients", "Common climate coefficient"]:
    records = [r for r in national_records if r["policy"] == policy]
    assert len(records) == 23
    complements = sum(int(r["biodiversity_complement"]) for r in records)
    land_value = math.fsum(float(r["E_USD2015_per_year"]) for r in records)
    transfers = math.fsum(float(r["net_transfer_USD2015_per_year"]) for r in records)
    print(f"{policy}: {complements} complements; land value {land_value/1e9:.3f}; transfers {transfers/1e9:.3f} billion USD2015/year")
from IPython.display import display, Markdown, Image
for path in sorted((output / "tables").glob("*.md")):
    display(Markdown(path.read_text()))


## Full source-coefficient calculation

Reconstruct and verify the 104,300,544-byte European database, then evaluate every original action. The calculation checks all recorded policy assignments, all three payoff-choice bands, dominance witnesses, support regions, resource bounds and exclusion certificates. Exact rational upper envelopes recover climate-only coverage across every nonnegative coefficient, including its unbounded tail. Optimistic coordinate checks also evaluate the full response sets at finite breakpoints for the four primary climate-only exclusions.

This is coefficient-level reproduction on the fixed published technology. It is separate from uncertainty about ecological parameters and from observed landholder behavior. The calculation replays verifiable policy witnesses and certificates without rerunning the original policy-search optimizers.
The Belgian floor sensitivity is a targeted change to a national economic requirement, not a perturbation of the ecological production coefficients.


In [ ]:
RUN_FULL_SOURCE_REPLAY = False  # Change to True for full coefficient-level reproduction.
if RUN_FULL_SOURCE_REPLAY:
    started = time.perf_counter()
    run("code/verify_matrices.py", "--output", output / "matrix_replay")
    source_report = json.loads((output / "matrix_replay/verification.json").read_text())
    for field in ["policy_rows_checked", "policy_band_checks", "dominance_witness_pairs_verified",
                  "scalar_support_segments_checked", "resource_certificates_checked"]:
        print(field.replace("_", " ") + ":", source_report[field])
    print(f"Full source calculation: {time.perf_counter() - started:.1f} seconds")
else:
    print("Quick mode uses the released numerical certificates. Full source replay is disabled.")


## Shared climate coefficients and the binding national safeguard

A common coefficient must satisfy every participating country's safeguards. The Belgian certificate excludes every climate coefficient at least 0.269, even with an unrestricted nonnegative biodiversity coefficient. Requiring one admissible low climate coefficient changes how many countries can be served without biodiversity payments.

The calculation enumerates the finite coverage patterns and verifies complete constructions. At the primary ambition, the lower and upper bounds coincide for one through seven climate coefficients under every specified choice band. The lower-target records retain cases where widening the choice band breaks a construction.


In [ ]:
price_result_dir = output / "price_groups"
if not (price_result_dir / "price_group_verification.json").exists():
    price_result_dir.mkdir(parents=True, exist_ok=True)
    for record in (root / "data/price_groups").iterdir():
        if record.is_file():
            shutil.copyfile(record, price_result_dir / record.name)
    print("Using released policy certificates; full mode independently reconstructs them from source.")
shared = json.loads((output / "price_groups/price_group_verification.json").read_text())
ceiling = json.loads((output / "price_groups/climate_ceiling_certificate.json").read_text())
print("Complete constructions:", shared["complete_constructions"])
print("Country-policy response checks:", shared["response_checks"])
print("Belgian exclusion threshold:", ceiling["climate_cutoff"])
print(f"Certified shortfall above that threshold: {ceiling['certified_land_value_shortfall']:,.2f} USD2015/year")
print("Climate coefficients | Minimum biodiversity complements | Total nominal coefficients")
for row in shared["primary_frontier"]:
    q, k = row["maximum_climate_coefficients"], row["complement_lower_bound"]
    assert row["complement_upper_bound"] == k
    print(q, k, q+k, sep=" | ")
with (output / "price_groups/price_group_policy_checks.csv").open(newline="") as stream:
    checks = list(csv.DictReader(stream))
failed = [r for r in checks if r["compliant"] == "False"]
assert not any(float(r["ambition"]) == .05 for r in failed)
print("Retained lower-target response-band failures:", len(failed))
economic = json.loads((output / "price_groups/belgium_economic_ceiling.json").read_text())
print(f"At climate coefficient {economic['climate_coefficient']:.6f}, the maximum guaranteed Belgian land value preserving B and C is {economic['maximum_guaranteed_E_subject_to_B_C']/1e6:.3f} million USD2015/year.")
print(f"Required relaxation of its land-value floor: {100*economic['fraction_of_original_E_floor']:.2f}%.")
run("code/tables.py", "--data", output / "data", "--output", output / "tables")
display(Markdown((output / "tables/Climate_coefficient_frontier.md").read_text()))
display(Markdown((output / "tables/Table_S3.md").read_text()))
print("Belgian safeguard sensitivity is reported separately from the primary frontier.")


## Sensitivity of the Belgian economic safeguard

We reduce only Belgium's modeled land-value floor by 0%, 0.5%, 1%, 3% or 5%, while retaining every ecological requirement and the other countries' safeguards. The 1% intervals are unresolved integer optimization bounds rather than confidence intervals. The released table can be checked quickly; the full source run separately rebuilds the exact support regions and verifies all national responses.


In [ ]:
run("tests/test_safeguard.py", "--output", output / "safeguard_quick.json")
with (root / "data/safeguard_frontier.csv").open(newline="") as stream:
    safeguard_rows = list(csv.DictReader(stream))
print("Belgian reduction | q | Complement bounds | q + complement bounds")
for row in safeguard_rows:
    if float(row["Belgian_floor_relaxation"]) in (0.0, .01, .03):
        q = int(row["q"])
        lo, hi = int(row["lower_bound"]), int(row["upper_bound"])
        label = str(lo) if lo == hi else f"{lo}–{hi}"
        nominal = str(q+lo) if lo==hi else f"{q+lo}–{q+hi}"
        print(f"{100*float(row['Belgian_floor_relaxation']):g}% | {q} | {label} | {nominal}")


In [ ]:
if RUN_FULL_SOURCE_REPLAY:
    started = time.perf_counter()
    run("code/safeguard_sensitivity.py", "--repository", root,
        "--intervals", output / "matrix_replay/exact_climate_only_intervals.json",
        "--output", output / "safeguard")
    run("code/verify_safeguard.py", "--repository", root,
        "--results", output / "safeguard",
        "--output", output / "safeguard/independent_verification.json")
    report = json.loads((output / "safeguard/independent_verification.json").read_text())
    assert report["status"] == "verified" and report["long_double_response_checks"] == 483
    print(f"Original-coefficient safeguard reproduction: {time.perf_counter()-started:.1f} seconds")
else:
    print("Full original-coefficient safeguard reproduction is available in full mode.")


## Two fixed country groups

Assigning one climate coefficient to CEE-11 and another to the other twelve countries requires exactly 19 biodiversity complements at the primary ambition. Freely assigned groups require 14 with the same cap of two coefficients. All original national safeguards remain in place. The fixed partition is a descriptive restriction, not an optimized regional design.


In [ ]:
fixed = json.loads((output / "data/fixed_groups/fixed_group_verification.json").read_text())
assert fixed["complement_lower_bound"] == fixed["complement_upper_bound"] == 19
print("Fixed groups: minimum biodiversity complements", fixed["complement_lower_bound"])
for row in fixed["coverage_bounds"]:
    print(row["group"], "maximum climate-only coverage:", row["maximum_climate_only_coverage"])
print("Country-response checks:", fixed["response_checks"])
print("All original national floors retained:", fixed["all_original_national_floors_retained"])
print("Source calculation performed:", RUN_FULL_SOURCE_REPLAY)


## Publication figures

The figure script rebuilds every display from regenerated numerical tables. SVG, PNG and TIFF use a common canvas. The TIFF and PNG exports contain identical plotted pixels. Figure captions remain in the article and Supplement.


In [ ]:
run("code/figures.py", "--data", output / "data", "--output", output / "figures")
for path in sorted((output / "figures").glob("*.png")):
    display(Markdown("### " + path.stem.replace("_", " ")))
    display(Image(filename=str(path), width=850))


## Numerical regression and download

Compare the regenerated numerical records with the independent reference. Integer counts and logical classifications must agree exactly. Floating-point comparisons use a relative tolerance of 5 × 10⁻¹² and an absolute tolerance of 2 × 10⁻¹². The archive contains the recalculated reader tables, coefficient-level evidence and figures; it omits the large source database.


In [ ]:
run("tests/test_regression.py", "--output", output, *(["--full"] if RUN_FULL_SOURCE_REPLAY else []))
summary = {"source_coefficient_replay": RUN_FULL_SOURCE_REPLAY,
           "figure_generation": True, "safeguard_full_reproduction": RUN_FULL_SOURCE_REPLAY,
           "coefficient_sha256": "f82ee0744c2eb30c6b72f5fbc993a42645ebb09264917f9b940198a02c36497d"}
(output / "reproduction.json").write_text(json.dumps(summary, indent=2) + "\n")
archive = Path(shutil.make_archive(str(root / "outputs"), "zip", output))
with zipfile.ZipFile(archive) as check:
    assert check.testzip() is None
print("Output archive:", archive.name)
if in_colab:
    from google.colab import files
    files.download(str(archive))
else:
    from IPython.display import FileLink
    display(FileLink(os.path.relpath(archive, Path.cwd())))
